# Team 04 Tool Dev Notebook

This notebook is the deterministic development harness for the active Team 04 Python tool surface.

Use it when changing geometry helpers, tool schemas, placement heuristics, or graph-backed shape serialization before involving the live LLM path.

## Why this notebook exists

The current runtime already follows the supervision pattern well: planner -> central_reason -> tool spokes.

The practical improvement here is workflow separation:
1. deterministic tool-development in this notebook
2. live-LLM end-to-end validation in the companion notebook

The next code-level improvement after this split is a wing-targeted edit tool layer keyed by the stable graph indices.

In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (
    workspace_root,
    workspace_root.parent,
    workspace_root / "team_04",
    workspace_root.parent / "team_04",
)
TEAM_ROOT = next((path for path in candidate_roots if (path / "agent").exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError(
        "Run this notebook from the workspace root, the team_04 folder, or the team_04/test_notebooks folder."
    )

team_root_str = str(TEAM_ROOT)
if team_root_str not in sys.path:
    sys.path.insert(0, team_root_str)

DEV_PAYLOAD_PATH = TEAM_ROOT / "test_notebooks" / "tool_dev_mode_payload.json"
DEV_PAYLOAD_PATH

WindowsPath('C:/Users/baoqt/OneDrive/Documents/GitHub/AIA26_Studio/team_04/test_notebooks/tool_dev_mode_payload.json')

In [3]:
from agent.mcp_client import build_default_local_tool_client
from agent.tool_catalog import ToolCatalog
from agent.tools.generate_building_boundary import generate_building_boundary
from agent.tools.modify_building_boundary import modify_building_boundary

import plotly.graph_objects as go
from topologicpy.Edge import Edge
from topologicpy.Graph import Graph
from topologicpy.Plotly import Plotly
from topologicpy.Vertex import Vertex

tool_client = build_default_local_tool_client()
catalog = ToolCatalog.from_discovered_tools(tool_client.list_tools())

action_names = (
    "read_site",
    "generate_shape",
    "check_requested_position",
    "check_constraints",
    "optimize",
    "evaluate",
    "place_building",
    "analyze_remaining_positions",
)

{action: list(catalog.names_for_action(action)) for action in action_names}

{'read_site': ['analyze_site_boundary'],
 'generate_shape': ['generate_building_boundary'],
 'check_requested_position': ['remaining_buildable_positions',
  'requested_position_checker',
  'measure_boundary_proximity'],
 'check_constraints': [],
 'optimize': ['modify_building_boundary', 'modify_building_wings'],
 'evaluate': [],
 'place_building': ['import_building_boundary'],
 'analyze_remaining_positions': ['remaining_buildable_positions',
  'requested_position_checker',
  'measure_boundary_proximity']}

In [4]:
def _to_xyz(point):
    if len(point) >= 3:
        return (float(point[0]), float(point[1]), float(point[2]))
    return (float(point[0]), float(point[1]), 0.0)


def _dedupe_boundary(boundary):
    cleaned = []
    for point in boundary:
        xyz = _to_xyz(point)
        if not cleaned or any(abs(cleaned[-1][index] - xyz[index]) > 1e-6 for index in range(3)):
            cleaned.append(xyz)
    if len(cleaned) > 1 and all(abs(cleaned[0][index] - cleaned[-1][index]) <= 1e-6 for index in range(3)):
        cleaned = cleaned[:-1]
    return cleaned


def _normalize_points(points):
    normalized = []
    for point in points or []:
        if isinstance(point, (list, tuple)) and point and isinstance(point[0], (list, tuple)):
            normalized.extend(_normalize_points(point))
            continue
        normalized.append(_to_xyz(point))
    return normalized


def _graph_from_centerlines(centerline_graph):
    nodes = centerline_graph.get("nodes", [])
    edges = centerline_graph.get("edges", [])
    if not nodes or not edges:
        return None

    vertices = [Vertex.ByCoordinates(*_to_xyz(node["point"])) for node in nodes]
    graph_edges = [
        Edge.ByVertices(vertices[item["from_node_index"]], vertices[item["to_node_index"]])
        for item in edges
    ]
    graph_edges = [edge for edge in graph_edges if edge is not None]
    if not graph_edges:
        return None
    return Graph.ByVerticesEdges(vertices, graph_edges)


def _apply_zoom_to_fit(figure):
    figure.update_yaxes(scaleanchor="x", scaleratio=1, visible=False)
    figure.update_xaxes(visible=False)
    figure.update_layout(
        margin=dict(l=0, r=0, t=48, b=0),
        plot_bgcolor="#384128",
        paper_bgcolor="#384128",
        font=dict(color="#f8fafc", size=13),
    )
    return figure


def _add_boundary_trace(figure, boundary, *, line_color, line_width, fillcolor=None, name=None):
    points = _dedupe_boundary(boundary)
    if len(points) < 2:
        return
    xs = [point[0] for point in points] + [points[0][0]]
    ys = [point[1] for point in points] + [points[0][1]]
    figure.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            line=dict(color=line_color, width=line_width),
            fill="toself" if fillcolor else None,
            fillcolor=fillcolor,
            name=name,
            hoverinfo="skip",
            showlegend=False,
        )
    )


def _add_wing_labels(figure, wings):
    if not wings:
        return

    figure.add_trace(
        go.Scatter(
            x=[wing["centroid"][0] for wing in wings],
            y=[wing["centroid"][1] for wing in wings],
            mode="text",
            text=[f"wing {wing['wing_index']}" for wing in wings],
            textposition="middle center",
            textfont=dict(color="#f8fafc", size=13),
            hoverinfo="skip",
            showlegend=False,
        )
    )


def _add_centerline_graph_trace(figure, building_graph, *, pivot_points=None):
    centerline_graph = building_graph.get("centerline_graph", {})
    nodes = centerline_graph.get("nodes", [])
    edges = centerline_graph.get("edges", [])
    adjacency_list = centerline_graph.get("adjacency_list", [])
    if not nodes or not edges:
        return

    for edge in edges:
        start = nodes[edge["from_node_index"]]["point"]
        end = nodes[edge["to_node_index"]]["point"]
        midpoint_x = (start[0] + end[0]) / 2.0
        midpoint_y = (start[1] + end[1]) / 2.0
        figure.add_trace(
            go.Scatter(
                x=[start[0], end[0]],
                y=[start[1], end[1]],
                mode="lines",
                line=dict(color="#ff2d2d", width=4),
                hoverinfo="skip",
                showlegend=False,
            )
        )
        figure.add_trace(
            go.Scatter(
                x=[midpoint_x],
                y=[midpoint_y],
                mode="text",
                text=[f"e{edge['edge_index']} / wing {edge['wing_index']}"],
                textposition="top center",
                textfont=dict(color="#ffd166", size=12),
                hoverinfo="skip",
                showlegend=False,
            )
        )

    figure.add_trace(
        go.Scatter(
            x=[node["point"][0] for node in nodes],
            y=[node["point"][1] for node in nodes],
            mode="markers+text",
            marker=dict(color="#111827", size=10),
            text=[
                f"n{node['node_index']} (deg {len(adjacency_list[node['node_index']]) if node['node_index'] < len(adjacency_list) else 0})"
                for node in nodes
            ],
            textposition="top center",
            textfont=dict(color="#e5e7eb", size=12),
            hovertext=[f"node {node['node_index']}" for node in nodes],
            hoverinfo="text",
            showlegend=False,
        )
    )

    normalized_pivots = _normalize_points(pivot_points)
    if normalized_pivots:
        figure.add_trace(
            go.Scatter(
                x=[point[0] for point in normalized_pivots],
                y=[point[1] for point in normalized_pivots],
                mode="markers+text",
                marker=dict(color="#fde047", size=16, symbol="star"),
                text=[f"pivot {index}" for index, _ in enumerate(normalized_pivots)],
                textposition="bottom center",
                textfont=dict(color="#fde047", size=12),
                hoverinfo="skip",
                showlegend=False,
            )
        )


def make_plan_figure(site_boundary, boundary, wings, *, title, building_graph=None, pivot_points=None):
    figure = go.Figure()
    _add_boundary_trace(
        figure,
        site_boundary,
        line_color="#2563eb",
        line_width=4,
        name="site",
    )
    _add_boundary_trace(
        figure,
        boundary,
        line_color="#ea580c",
        line_width=3,
        fillcolor="rgba(249, 115, 22, 0.18)",
        name="building",
    )

    wing_palette = [
        ("#ef4444", "rgba(239, 68, 68, 0.18)"),
        ("#22c55e", "rgba(34, 197, 94, 0.18)"),
        ("#8b5cf6", "rgba(139, 92, 246, 0.18)"),
        ("#f59e0b", "rgba(245, 158, 11, 0.18)"),
        ("#06b6d4", "rgba(6, 182, 212, 0.18)"),
    ]
    for index, wing in enumerate(wings):
        line_color, fill_color = wing_palette[index % len(wing_palette)]
        _add_boundary_trace(
            figure,
            wing["boundary"],
            line_color=line_color,
            line_width=2,
            fillcolor=fill_color,
            name=f"wing {wing['wing_index']}",
        )

    _add_wing_labels(figure, wings)

    if building_graph is not None:
        _add_centerline_graph_trace(figure, building_graph, pivot_points=pivot_points)

    figure.update_layout(title=title)
    return _apply_zoom_to_fit(figure)


def make_graph_figure(building_graph, *, title, pivot_points=None):
    figure = go.Figure()
    _add_centerline_graph_trace(figure, building_graph, pivot_points=pivot_points)
    if not figure.data:
        raise ValueError("building_graph.centerline_graph does not contain any drawable edges")
    figure.update_layout(title=title)
    return _apply_zoom_to_fit(figure)

In [21]:
import importlib

building_shape_graph_module = importlib.import_module("agent.tools.building_shape_graph")
placement_optimizer_module = importlib.import_module("agent.tools.placement_optimizer")
generate_building_boundary_module = importlib.import_module("agent.tools.generate_building_boundary")
building_shape_graph_module = importlib.reload(building_shape_graph_module)
placement_optimizer_module = importlib.reload(placement_optimizer_module)
generate_building_boundary_module = importlib.reload(generate_building_boundary_module)

SITE_BOUNDARY = [
    [0.0, 0.0, 0.0],
    [84.0, 0.0, 0.0],
    [104.0, 24.0, 0.0],
    [88.0, 62.0, 0.0],
    [20.0, 56.0, 0.0],
    [0.0, 28.0, 0.0],
    [0.0, 0.0, 0.0],
]

generation_result = generate_building_boundary_module.generate_building_boundary(
    area=900.0,
    building_type="U",
    building_depth=18.0,
    shape_ratio=0.62,
    site_boundary=SITE_BOUNDARY,
    optimize_placement=True,
    placement_clearance=2.0,
    population_size=40,
    generation_count=40,
    random_seed=11,
)

DEV_PAYLOAD_PATH.write_text(json.dumps(generation_result, indent=2), encoding="utf-8")
data = generation_result["data"]

{
    "saved_to": str(DEV_PAYLOAD_PATH),
    "shape_type": data["shape_type"],
    "wing_count": len(data["wings"]),
    "adjacency_list": data["building_graph"]["adjacency_list"],
    "centerline_node_count": data["building_graph"]["centerline_graph"]["node_count"],
    "centerline_edge_count": data["building_graph"]["centerline_graph"]["edge_count"],
    "placement_optimization": data["placement_optimization"],
    "site_fit_summary": data["site_fit_summary"],
}

{'saved_to': 'C:\\Users\\baoqt\\OneDrive\\Documents\\GitHub\\AIA26_Studio\\team_04\\test_notebooks\\tool_dev_mode_payload.json',
 'shape_type': 'U',
 'wing_count': 3,
 'adjacency_list': [[1, 2], [0], [0]],
 'centerline_node_count': 4,
 'centerline_edge_count': 3,
 'placement_optimization': {'optimized': True,
  'centroid_xy': [56.64291, 29.750864],
  'rotation_degrees': 3.808256,
  'objective': -17.463717,
  'outside_area_sqm': 0.0,
  'clearance_m': 17.463717,
  'fits_within_site_boundary': True,
  'population_size': 40,
  'generation_count': 40,
  'random_seed': 11,
  'target_location_xy': [],
  'selected_option_id': 'placement_option_01',
  'saved_option_count': 5,
  'saved_options': [{'option_id': 'placement_option_01',
    'label': 'Placement option 1',
    'status': 'selected',
    'centroid_xy': [56.64291, 29.750864],
    'rotation_degrees': 3.808256,
    'objective': -17.463717,
    'outside_area_sqm': 0.0,
    'clearance_m': 17.463717,
    'fits_within_site_boundary': True,
   

In [5]:
make_plan_figure(
    SITE_BOUNDARY,
    data["boundary"],
    data["wings"],
    title="Generated footprint with labeled wings, nodes, and centerline graph",
    building_graph=data["building_graph"],
)

In [10]:
import importlib
import json

building_shape_graph_module = importlib.import_module("agent.tools.building_shape_graph")
modify_building_wings_module = importlib.import_module("agent.tools.modify_building_wings")
tools_package = importlib.import_module("agent.tools")
mcp_client_module = importlib.import_module("agent.mcp_client")

building_shape_graph_module = importlib.reload(building_shape_graph_module)
modify_building_wings_module = importlib.reload(modify_building_wings_module)
tools_package = importlib.reload(tools_package)
mcp_client_module = importlib.reload(mcp_client_module)
tool_client = mcp_client_module.build_default_local_tool_client()

modified_result = json.loads(
    tool_client.call_tool(
        "modify_building_wings",
        {
            "geometry_id": data["geometry_id"],
            "shape_type": data["shape_type"],
            "wings": data["wings"],
            "building_graph": data["building_graph"],
            "edits": [
                {"wing_index": 2, "rotation_degrees": 180.0},
            ],
            "site_boundary": SITE_BOUNDARY,
            "clearance": 1.5,
        },
    )
)

{
    "tool_name": modified_result["metadata"]["tool_name"],
    "applied_edits": modified_result["data"]["applied_edits"],
    "fits_within_site_boundary": modified_result["data"]["fits_within_site_boundary"],
    "violations": modified_result["data"]["violations"],
    "centerline_edge_count": modified_result["data"]["building_graph"]["centerline_graph"]["edge_count"],
    "rotated_wing_centroid": modified_result["data"]["wings"][2]["centroid"],
}

Edge.ByStartVertexEndVertex - Error: The distance between the input vertexA and vertexB parameters is less than the input tolerance. Returning None.
caller name: ByVertices
Edge.ByVertices - Error: Could not create an edge. Returning None.
caller name: ByVertices
Wire.ByVertices - Warning: Degenerate edge. Skipping.
caller name: ByVertices
Face.ByVertices - Error: Could not create a closed base wire. Returning None.


{'tool_name': 'modify_building_wings',
 'applied_edits': [{'wing_index': 2,
   'thickness_scale': 1.0,
   'rotation_degrees': 180.0,
   'rotation_pivot': [23.667895, 26.171922, 0.0]}],
 'fits_within_site_boundary': True,
 'violations': [],
 'centerline_edge_count': 3,
 'rotated_wing_centroid': [23.667897, 28.341922, 0.0]}

In [11]:
pivot_points = [
    edit["rotation_pivot"]
    for edit in modified_result["data"]["applied_edits"]
    if abs(edit["rotation_degrees"]) > 1e-9
]

make_plan_figure(
    SITE_BOUNDARY,
    modified_result["data"]["boundary"],
    modified_result["data"]["wings"],
    title="End-wing rotation test with labels and highlighted rotation pivot",
    building_graph=modified_result["data"]["building_graph"],
    pivot_points=pivot_points,
)

In [ ]:
## Site boundary graph, preferred side alignment, and proximity line

This section uses a more difficult site with a diagonal side, aligns the building's largest edge to the user-preferred site side, and shows the thin before-and-after proximity lines to that side.

In [24]:
import importlib

PREFERRED_SIDE_LABEL = "side_2"

modify_building_boundary_module = importlib.import_module("agent.tools.modify_building_boundary")
site_boundary_graph_module = importlib.import_module("agent.tools.site_boundary_graph")
measure_boundary_proximity_module = importlib.import_module("agent.tools.measure_boundary_proximity")

modify_building_boundary_module = importlib.reload(modify_building_boundary_module)
site_boundary_graph_module = importlib.reload(site_boundary_graph_module)
measure_boundary_proximity_module = importlib.reload(measure_boundary_proximity_module)

site_graph_result = site_boundary_graph_module.analyze_site_boundary(SITE_BOUNDARY)
proximity_before_move = measure_boundary_proximity_module.measure_boundary_proximity(
    geometry_id=data["geometry_id"],
    building_boundary=data["boundary"],
    site_boundary=SITE_BOUNDARY,
    site_edge_label=PREFERRED_SIDE_LABEL,
)
move_toward_side_result = modify_building_boundary_module.modify_building_boundary(
    geometry_id=data["geometry_id"],
    boundary=data["boundary"],
    site_boundary=SITE_BOUNDARY,
    move_toward_site_edge_label=PREFERRED_SIDE_LABEL,
    align_largest_edge_to_site_edge=True,
    target_edge_clearance=10.0,
)
proximity_after_move = measure_boundary_proximity_module.measure_boundary_proximity(
    geometry_id=data["geometry_id"],
    building_boundary=move_toward_side_result["data"]["transformed_boundary"],
    site_boundary=SITE_BOUNDARY,
    site_edge_label=PREFERRED_SIDE_LABEL,
)

{
    "preferred_side": PREFERRED_SIDE_LABEL,
    "site_side_labels": [
        edge["label"]
        for edge in site_graph_result["data"]["site_boundary_graph"]["edges"]
    ],
    "site_corner_labels": [
        node["label"]
        for node in site_graph_result["data"]["site_boundary_graph"]["nodes"]
    ],
    "selected_side_before": proximity_before_move["data"]["selected_site_edge"],
    "selected_side_after": proximity_after_move["data"]["selected_site_edge"],
    "fits_within_site_boundary": move_toward_side_result["data"]["fits_within_site_boundary"],
    "move_transform": move_toward_side_result["data"]["transform_parameters"],
}

{'preferred_side': 'side_2',
 'site_side_labels': ['side_0',
  'side_1',
  'side_2',
  'side_3',
  'side_4',
  'side_5'],
 'site_corner_labels': ['corner_0',
  'corner_1',
  'corner_2',
  'corner_3',
  'corner_4',
  'corner_5'],
 'selected_side_before': {'edge_index': 2,
  'label': 'side_2',
  'cardinal_hint': 'east',
  'minimum_distance_m': 18.166332,
  'nearest_building_point': [76.50007, 42.498767, 0.0],
  'nearest_site_point': [93.242804, 49.548339, 0.0]},
 'selected_side_after': {'edge_index': 2,
  'label': 'side_2',
  'cardinal_hint': 'east',
  'minimum_distance_m': 10.0,
  'nearest_building_point': [81.171252, 52.448867, 0.0],
  'nearest_site_point': [90.387606, 56.329437, 0.0]},
 'fits_within_site_boundary': True,
 'move_transform': {'target_centroid_xy': None,
  'translate_by_xy': [0.0, 0.0],
  'rotation_degrees': 0.0,
  'orientation_degrees': 0.0,
  'applied_rotation_degrees': 0.0,
  'alignment_rotation_degrees': 19.025396,
  'rotation_origin_xy': [56.64291, 29.750864],
  'ap

In [25]:
site_graph = site_graph_result["data"]["site_boundary_graph"]
target_side = next(
    edge
    for edge in site_graph["edges"]
    if edge["label"] == PREFERRED_SIDE_LABEL
    )
target_start = site_graph["nodes"][target_side["from_node_index"]]["point"]
target_end = site_graph["nodes"][target_side["to_node_index"]]["point"]
selected_side_before = proximity_before_move["data"]["selected_site_edge"]
selected_side_after = proximity_after_move["data"]["selected_site_edge"]
aligned_building_edge = move_toward_side_result["data"]["transform_parameters"]["aligned_building_edge"]

figure = go.Figure()
_add_boundary_trace(
    figure,
    SITE_BOUNDARY,
    line_color="#2563eb",
    line_width=4,
    name="site",
)
_add_boundary_trace(
    figure,
    data["boundary"],
    line_color="#ea580c",
    line_width=3,
    fillcolor="rgba(249, 115, 22, 0.18)",
    name="original",
)
_add_boundary_trace(
    figure,
    move_toward_side_result["data"]["transformed_boundary"],
    line_color="#22c55e",
    line_width=3,
    fillcolor="rgba(34, 197, 94, 0.18)",
    name="moved",
)
figure.add_trace(
    go.Scatter(
        x=[target_start[0], target_end[0]],
        y=[target_start[1], target_end[1]],
        mode="lines+text",
        line=dict(color="#fde047", width=5, dash="dash"),
        text=[target_side["label"], ""],
        textposition="top center",
        textfont=dict(color="#fde047", size=12),
        hoverinfo="skip",
        showlegend=False,
    )
)
figure.add_trace(
    go.Scatter(
        x=[selected_side_before["nearest_building_point"][0], selected_side_before["nearest_site_point"][0]],
        y=[selected_side_before["nearest_building_point"][1], selected_side_before["nearest_site_point"][1]],
        mode="lines",
        line=dict(color="#fca5a5", width=1),
        hoverinfo="skip",
        showlegend=False,
    )
)
figure.add_trace(
    go.Scatter(
        x=[selected_side_after["nearest_building_point"][0], selected_side_after["nearest_site_point"][0]],
        y=[selected_side_after["nearest_building_point"][1], selected_side_after["nearest_site_point"][1]],
        mode="lines",
        line=dict(color="#bbf7d0", width=1),
        hoverinfo="skip",
        showlegend=False,
    )
)
figure.add_trace(
    go.Scatter(
        x=[aligned_building_edge["start_point"][0], aligned_building_edge["end_point"][0]],
        y=[aligned_building_edge["start_point"][1], aligned_building_edge["end_point"][1]],
        mode="lines",
        line=dict(color="#86efac", width=4),
        hoverinfo="skip",
        showlegend=False,
    )
)
figure.add_trace(
    go.Scatter(
        x=[node["point"][0] for node in site_graph["nodes"]],
        y=[node["point"][1] for node in site_graph["nodes"]],
        mode="markers+text",
        marker=dict(color="#f8fafc", size=9),
        text=[node["label"] for node in site_graph["nodes"]],
        textposition="bottom center",
        textfont=dict(color="#f8fafc", size=11),
        hoverinfo="skip",
        showlegend=False,
    )
)
figure.update_layout(title="Align the building main edge to the diagonal preferred side and show thin proximity lines")
_apply_zoom_to_fit(figure)

## Remaining-area seed for building 2

This section shows the current second-building seeding logic after the planner repair. It samples remaining feasible centroid candidates after placing building 1, then picks the nearest candidate to building 2's requested position.

In [6]:
import importlib
from IPython.display import display
import plotly.graph_objects as go

if "SITE_BOUNDARY" not in globals():
    SITE_BOUNDARY = [
        [0.0, 0.0, 0.0],
        [84.0, 0.0, 0.0],
        [104.0, 24.0, 0.0],
        [88.0, 62.0, 0.0],
        [20.0, 56.0, 0.0],
        [0.0, 28.0, 0.0],
        [0.0, 0.0, 0.0],
    ]

generate_building_boundary_module = importlib.reload(importlib.import_module("agent.tools.generate_building_boundary"))
decision_engine_module = importlib.reload(importlib.import_module("agent.decision_engine"))
building_shape_graph_module = importlib.reload(importlib.import_module("agent.tools.building_shape_graph"))
multi_building_mock_module = importlib.reload(importlib.import_module("agent.tools.multi_building_mock"))

if "data" not in globals():
    data = generate_building_boundary_module.generate_building_boundary(
        area=900.0,
        building_type="U",
        building_depth=18.0,
        shape_ratio=0.62,
        site_boundary=SITE_BOUNDARY,
        optimize_placement=True,
        placement_clearance=2.0,
        population_size=40,
        generation_count=40,
        random_seed=11,
    )["data"]

second_building_template = generate_building_boundary_module.generate_building_boundary(
    area=420.0,
    building_type="I",
    building_depth=12.0,
    shape_ratio=0.66,
    optimize_placement=False,
 )["data"]

first_building = {
    "geometry_id": data["geometry_id"],
    "boundary": data["boundary"],
}
second_building_requested_position = [92.0, 44.0]
remaining_result = multi_building_mock_module.mock_remaining_buildable_positions(
    site_boundary=SITE_BOUNDARY,
    placed_buildings=[first_building],
    candidate_building_boundary=second_building_template["boundary"],
    grid_size=6.0,
    clearance=2.0,
    max_positions=20,
 )
second_building_state = {
    "placed_buildings": [first_building],
    "remaining_candidate_positions": remaining_result["data"]["candidate_positions"],
    "requested_positions": [[30.0, 24.0], second_building_requested_position],
}
selected_seed = decision_engine_module._select_generation_location_hint(second_building_state)
seeded_second_building = second_building_template
if selected_seed is not None:
    seeded_model = building_shape_graph_module.build_shape_model(
        area=420.0,
        building_type="I",
        building_depth=12.0,
        shape_ratio=0.66,
    )
    seeded_model = building_shape_graph_module.apply_shape_transform(
        seeded_model,
        translation_xy=(selected_seed[0], selected_seed[1]),
    )
    seeded_second_building = building_shape_graph_module.serialize_shape_model(seeded_model)

display({
    "candidate_count": remaining_result["data"]["candidate_count"],
    "requested_position": second_building_requested_position,
    "selected_seed": selected_seed,
    "second_building_shape_type": second_building_template["shape_type"],
    "seeded_centerline_node_count": seeded_second_building["building_graph"]["centerline_graph"]["node_count"],
    "seeded_centerline_edge_count": seeded_second_building["building_graph"]["centerline_graph"]["edge_count"],
    "candidate_positions": remaining_result["data"]["candidate_positions"],
})

figure = go.Figure()
_add_boundary_trace(
    figure,
    SITE_BOUNDARY,
    line_color="#2563eb",
    line_width=4,
    name="site",
)
_add_boundary_trace(
    figure,
    data["boundary"],
    line_color="#ea580c",
    line_width=3,
    fillcolor="rgba(249, 115, 22, 0.18)",
    name="building 1",
)
_add_boundary_trace(
    figure,
    seeded_second_building["boundary"],
    line_color="#a855f7",
    line_width=2,
    fillcolor="rgba(168, 85, 247, 0.10)",
    name="building 2 at selected seed",
)
for wing in seeded_second_building["wings"]:
    _add_boundary_trace(
        figure,
        wing["boundary"],
        line_color="#c084fc",
        line_width=2,
        fillcolor="rgba(192, 132, 252, 0.14)",
        name=f"building 2 wing {wing['wing_index']}",
    )
_add_wing_labels(figure, seeded_second_building["wings"])
_add_centerline_graph_trace(figure, seeded_second_building["building_graph"])

candidate_positions = remaining_result["data"]["candidate_positions"]
figure.add_trace(
    go.Scatter(
        x=[point[0] for point in candidate_positions],
        y=[point[1] for point in candidate_positions],
        mode="markers",
        marker=dict(color="#93c5fd", size=9),
        hoverinfo="skip",
        showlegend=False,
    )
)
figure.add_trace(
    go.Scatter(
        x=[second_building_requested_position[0]],
        y=[second_building_requested_position[1]],
        mode="markers+text",
        marker=dict(color="#fde047", size=13, symbol="diamond"),
        text=["requested"],
        textposition="top center",
        textfont=dict(color="#fde047", size=12),
        hoverinfo="skip",
        showlegend=False,
    )
)
if selected_seed is not None:
    figure.add_trace(
        go.Scatter(
            x=[selected_seed[0]],
            y=[selected_seed[1]],
            mode="markers+text",
            marker=dict(color="#22c55e", size=16, symbol="star"),
            text=["selected seed"],
            textposition="bottom center",
            textfont=dict(color="#bbf7d0", size=12),
            hoverinfo="skip",
            showlegend=False,
        )
    )
figure.update_layout(title="Remaining-area candidates and selected seed for building 2")
_apply_zoom_to_fit(figure)

{'candidate_count': 12,
 'requested_position': [92.0, 44.0],
 'selected_seed': [69.0, 51.0],
 'second_building_shape_type': 'I',
 'seeded_centerline_node_count': 2,
 'seeded_centerline_edge_count': 1,
 'candidate_positions': [[21.0, 9.0, 0.0],
  [27.0, 9.0, 0.0],
  [33.0, 9.0, 0.0],
  [39.0, 9.0, 0.0],
  [45.0, 9.0, 0.0],
  [51.0, 9.0, 0.0],
  [51.0, 51.0, 0.0],
  [57.0, 9.0, 0.0],
  [57.0, 51.0, 0.0],
  [63.0, 9.0, 0.0],
  [63.0, 51.0, 0.0],
  [69.0, 51.0, 0.0]]}

## Next use

Use this notebook when you are iterating on tool contracts, geometry math, graph payloads, or placement heuristics.

Use the companion end-to-end notebook when you want the real planner plus supervisor plus LLM path.